# ICS 3202: Artificial Intelligence — Semester Project Deliverable 2 (CAT 2)
## LegalMatch AI — ML Engine: Base Model (Sentence-Transformer semantic retrieval)

**Group:** [Group name] · **Members:** [Member 1], [Member 2], [Member 3], [Member 4]

### What the application needs from the ML engine
A client types their legal problem in everyday language (e.g. *"my landlord locked my shop because I was late with rent"*).
The engine must (1) find past Kenyan court cases that are about the same kind of problem, and
(2) pull out the advocates who argued those cases, so the app can suggest lawyers with relevant experience.

### Notebook map
| Section | Question | What it covers | Presenter |
|---|---|---|---|
| 1 | Q1 | ML approach and algorithms | [Member 1] |
| 2 | Q2 | Loading, data cleaning, EDA (7 observations), final modelling dataset | [Member 2] |
| 3 | Q3 | Base model + validity checks (cross-validation, regularisation, retrieval metrics, threshold) | [Member 3] |
| 4 | Q4 | Further validation: comparing other models through the same pipeline | [Member 4] |

> **How to run:** Google Colab → *Runtime → Change runtime type → T4 GPU* → *Run all*. CPU also works, just slower (see `CONFIG` below to shrink the sample).

## 1. ML Approach and Algorithms (Q1)

**Type of ML problem.** This is *unsupervised / self-supervised NLP*: our dataset has no "correct lawyer" labels, so we cannot train a classifier that predicts a lawyer directly.
Instead we treat matching as **information retrieval** followed by **information extraction**.

| Stage | Job | Base algorithm (this notebook) | Alternatives compared in Q4 |
|---|---|---|---|
| A. Semantic retrieval | Find the case passages most similar in *meaning* to the client's description | **Sentence-Transformer `all-MiniLM-L6-v2`** (dense embeddings + cosine similarity) | TF-IDF, BM25, `multi-qa-MiniLM-L6-cos-v1`, `all-mpnet-base-v2`, hybrid BM25 + MiniLM |
| B. Lawyer extraction | Pull advocate names and which party they represented from the matched judgments | **Rule-based pattern extractor** for Kenyan appearance lines (*"Mr. Otieno for the claimant"*, *"Kamau & Co. Advocates"*) | spaCy `en_core_web_sm` PERSON entities |
| C. Ranking | Turn matched cases into a ranked list of lawyers | Sum of case-similarity scores per advocate (state prosecutors removed) | — |

**Why a sentence-transformer?** Clients and judges use different words for the same thing (*"fired"* vs *"unfair termination"*). Keyword methods only match exact words; embeddings place sentences with similar meaning close together, so they can bridge that vocabulary gap. MiniLM is small (384-dimension vectors, ~80 MB), which suits a mobile-backed app.

**Why a classifier appears in Q3.** We derive a *practice-area label* (employment, land, criminal, …) for each judgment from the court and case type. That label is not what the app predicts; it is a yardstick that lets us **measure** whether the embeddings group legal problems correctly, using cross-validation and regularisation.

*(This summary is what we entered in the shared semester-groups spreadsheet.)*

### Setup

In [ ]:
# Install the libraries Colab does not ship with
!pip install -q datasets sentence-transformers rank_bm25 spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import re, json, ast, time, unicodedata, warnings, os
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, f1_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 110)
sns.set_theme(style='whitegrid')
PALETTE = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300']   # fixed colour order used in every chart

# ---- CONFIG: every tunable number lives here so the run is reproducible ----
CONFIG = {
    'SEED': 42,
    'MIN_DOC_WORDS': 50,          # judgments shorter than this are stubs (headers / "judgment reserved")
    'PASSAGE_WORDS': 160,         # passage size for the embedder (MiniLM reads at most 256 word-pieces)
    'PASSAGE_OVERLAP': 30,        # words shared between neighbouring passages so no sentence is cut in half
    'MAX_DOCS_PER_AREA': 150,     # modelling sample: judgments per practice area (raise on GPU)
    'MAX_PASSAGES_PER_DOC': 25,   # cap so one very long judgment cannot dominate the index
    'BASE_MODEL': 'sentence-transformers/all-MiniLM-L6-v2',
    'RUN_FULL_COMPARISON': True,  # Q4: also encode with the two extra transformer models
}
SEED = CONFIG['SEED']
rng = np.random.default_rng(SEED)
os.makedirs('artifacts', exist_ok=True)

## 2. Data Cleaning and Exploratory Data Analysis (Q2)

**Dataset:** [`MakiniAI/kenya-law-corpus`](https://huggingface.co/datasets/MakiniAI/kenya-law-corpus) — Kenyan legislation and court judgments scraped from Kenya Law (National Council for Law Reporting), licensed CC BY-NC 4.0.
According to the dataset card, documents were split into **800-word chunks with a 100-word overlap** — we verify both claims below because they directly affect modelling.

### 2.1 Load and inspect the raw data

In [ ]:
ds = load_dataset("MakiniAI/kenya-law-corpus", split="train")
raw = ds.to_pandas()

print("Raw shape (rows, columns):", raw.shape)
print("\nColumns and types:")
print(raw.dtypes)
raw.head(3)

### 2.2 Flatten the schema
Some versions of the dataset keep fields such as `title`, `url` and `type` inside a JSON `metadata` column, others store them as top-level columns.
We handle both, so the rest of the notebook always sees the same columns: `text, title, url, type, section, chunk_id`.

In [ ]:
def parse_meta(m):
    # metadata may arrive as a dict, a JSON string, a Python-literal string, or be missing
    if isinstance(m, dict):
        return m
    if isinstance(m, str) and m.strip():
        try:
            return json.loads(m)
        except json.JSONDecodeError:
            try:
                return ast.literal_eval(m)
            except Exception:
                return {}
    return {}

def is_blank(s):
    return s.isna() | s.astype(str).str.strip().isin(['', 'nan', 'None', 'null', 'NaN'])

df = raw.copy()
if 'metadata' in df.columns:
    meta = pd.json_normalize(df['metadata'].apply(parse_meta).tolist())
    meta.index = df.index
    for col in meta.columns:
        if col in df.columns:   # keep the top-level value, fill its gaps from metadata
            df[col] = df[col].where(~is_blank(df[col]), meta[col])
        else:
            df[col] = meta[col]
    df = df.drop(columns=['metadata'])

for col in ['text', 'title', 'url', 'type', 'section', 'chunk_id']:
    if col not in df.columns:
        df[col] = np.nan

print("Columns after flattening:", list(df.columns))

### 2.3 Data-quality audit (before cleaning)
We count missing *and* empty-string values, because scraped data often stores "nothing" as `""` rather than `NaN` — `isnull()` alone would miss it.

In [ ]:
def missing_report(frame):
    return pd.Series({c: round(100 * is_blank(frame[c]).mean(), 2) for c in frame.columns}, name='% missing or empty')

audit_before = missing_report(df)
print(audit_before.sort_values(ascending=False).to_string())
print("\nDistinct values of 'type':", df['type'].astype(str).str.strip().str.lower().value_counts().to_dict())

### 2.4 Cleaning steps
Every step records how many rows it removed and *why*, so the full path from raw data to modelling data is auditable (summary table in 2.8).

In [ ]:
cleaning_log = []
def log_step(step, before, after, why):
    cleaning_log.append({'step': step, 'rows_before': before, 'rows_after': after,
                         'removed': before - after, 'reason': why})
    print(f"{step:<45} {before:>8,} -> {after:>8,}  (removed {before - after:,})")

# Step 1 - standardise the document type; fall back to the URL when 'type' is missing
df['type'] = (df['type'].astype(str).str.strip().str.lower()
              .str.replace(r'[\s\-]+', '_', regex=True)
              .replace({'judgment': 'case_law', 'judgement': 'case_law', 'caselaw': 'case_law', 'act': 'legislation'}))
url_type = np.select([df['url'].astype(str).str.contains('/judgment/'), df['url'].astype(str).str.contains('/act/')],
                     ['case_law', 'legislation'], default='unknown')
df['type'] = df['type'].where(df['type'].isin(['case_law', 'legislation']), url_type)

# Step 2 - normalise text: unicode (curly quotes, non-breaking spaces), control characters, repeated whitespace.
# We do NOT lowercase or remove stop-words: the transformer was trained on natural sentences and uses them for meaning.
def clean_text(t):
    if t is None or (isinstance(t, float) and np.isnan(t)):
        return ''
    t = unicodedata.normalize('NFKC', str(t))
    t = re.sub(r'[\x00-\x08\x0b-\x1f\x7f]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df['text'] = df['text'].apply(clean_text)
df['title'] = df['title'].apply(clean_text)
before = len(df); df = df[df['text'].str.len() > 0].copy()
log_step('Drop chunks with empty text', before, len(df), 'Nothing to embed or search')

# Step 3 - build a document id and the chunk's position inside its document
df['doc_id'] = df['url'].where(~is_blank(df['url']), df['chunk_id'].astype(str).str.split('##').str[0])
pos_from_section = df['section'].astype(str).str.extract(r'(\d+)\s*$')[0]
pos_from_id = df['chunk_id'].astype(str).str.extract(r'chunk_?(\d+)')[0]
df['chunk_idx'] = pd.to_numeric(pos_from_section.fillna(pos_from_id), errors='coerce')
print(f"Chunks with no recoverable position: {df['chunk_idx'].isna().mean():.1%} (falling back to file order)")
df['chunk_idx'] = df['chunk_idx'].fillna(df.groupby('doc_id').cumcount())

# Step 4 - duplicates
before = len(df); df = df.drop_duplicates(subset=['doc_id', 'chunk_idx'])
log_step('Drop repeated (document, position) rows', before, len(df), 'Same chunk scraped twice')
before = len(df); df = df[~df.duplicated(subset=['text'], keep='first')]
log_step('Drop exact duplicate chunk text', before, len(df),
         'A judgment listed under two URLs would be retrieved twice and double-count its lawyers')

df['chunk_words'] = df['text'].str.split().str.len()
chunks = df.copy()      # kept for the chunk-level EDA below

### 2.5 Verify the chunk overlap and rebuild whole documents
If neighbouring chunks really repeat 100 words, two problems follow: the same sentence is indexed twice (inflating similarity for long judgments), and a lawyer's name in the overlap is counted twice.
We **measure** the overlap between each chunk and the next one, strip it, and stitch each judgment back together. This also lets us re-split it later into passages sized for our model (2.7 and 3.1).

In [ ]:
def overlap_len(prev_words, cur_words, max_k=150, min_k=10):
    # longest k such that the last k words of the previous chunk == the first k words of this chunk
    for k in range(min(max_k, len(prev_words), len(cur_words)), min_k - 1, -1):
        if prev_words[-k:] == cur_words[:k]:
            return k
    return 0

extra_cols = [c for c in ['court', 'date', 'citation'] if c in df.columns]
records, overlaps = [], []
for doc_id, g in df.sort_values(['doc_id', 'chunk_idx']).groupby('doc_id', sort=False):
    words, prev = [], None
    for t in g['text']:
        w = t.split()
        if prev is not None:
            k = overlap_len(prev, w)
            overlaps.append(k)
            words.extend(w[k:])
        else:
            words.extend(w)
        prev = w
    first = g.iloc[0]
    rec = {'doc_id': doc_id, 'type': first['type'], 'title': first['title'], 'url': first['url'],
           'n_chunks': len(g), 'text': ' '.join(words), 'n_words': len(words)}
    for c in extra_cols:
        rec[c] = first[c]
    records.append(rec)

docs = pd.DataFrame(records)
overlaps = np.array(overlaps)
log_step('Merge chunks into whole documents', len(df), len(docs), 'One row per judgment / Act; overlap removed')

before = len(docs); docs = docs[docs['n_words'] >= CONFIG['MIN_DOC_WORDS']].copy()
log_step(f"Drop documents < {CONFIG['MIN_DOC_WORDS']} words", before, len(docs),
         'Stubs such as headers or "ruling reserved" notices carry no facts to match on')

### 2.6 Feature engineering: court, year, practice area, language, advocates
Most of the structure we need is hidden in the Kenya Law **URL** and the case **title**:

* URL `…/akn/ke/judgment/keelrc/2023/1234/eng@2023-05-12` → court code `keelrc` (Employment & Labour Relations Court), year 2023, date.
* Title `John Doe v ABC Ltd (Cause E123 of 2021) [2023] KEELRC 1234 (KLR)` → case type *Cause* and the neutral citation.

**Practice-area label (our evaluation yardstick).** We assign an area only when there is a reliable signal: first the *specialised court* (Kenya has dedicated employment, land and tax courts), then the *case type in the title* (e.g. *Criminal Appeal*, *Succession Cause*, *Petition*). Mixed-bag case types (plain *Civil Appeal*) are labelled **General Civil**, and anything else **Unclassified**. Both stay in the search index as realistic distractors but are excluded from the supervised checks, because they are not one coherent area.

In [ ]:
COURTS = {   # Kenya Law neutral-citation court codes
    'kesc': 'Supreme Court', 'keca': 'Court of Appeal', 'kehc': 'High Court',
    'keelrc': 'Employment & Labour Relations Court', 'keic': 'Industrial Court (pre-2012)',
    'keelc': 'Environment & Land Court', 'kemc': "Magistrates' Court", 'kekc': "Kadhi's Court",
    'ketat': 'Tax Appeals Tribunal', 'kebprt': 'Business Premises Rent Tribunal',
}
COURT_AREA = {'keelrc': 'Employment & Labour', 'keic': 'Employment & Labour', 'keelc': 'Land & Property',
              'kebprt': 'Land & Property', 'ketat': 'Tax', 'kekc': 'Family & Succession'}
TITLE_RULES = [   # order matters: the first match wins
    ('Constitutional & Administrative', r'judicial review|ex[\s\-]?parte|constitutional|human rights|\bpetition\b'),
    ('Criminal',            r'criminal|\brepublic v\b|\bv\.? republic\b|murder|robbery|sexual offences|defilement|\bbail\b'),
    ('Family & Succession', r'succession|probate|estate of|divorce|matrimonial|custody|children case|adoption|family'),
    ('Land & Property',     r'environment (?:and|&) land|\belc\b|land case|land appeal|tenancy|rent'),
    ('Employment & Labour', r'employment|labour|industrial cause'),
    ('Tax',                 r'\btax\b|income tax|vat appeal'),
    ('Commercial & Contract', r'commercial|winding up|insolvency|bankruptcy|arbitration|companies'),
    ('General Civil',       r'civil appeal|civil suit|civil case|civil application|\bhccc\b|miscellaneous'),
]

def court_year_date(row):
    url, title = str(row['url']), str(row['title'])
    code_, year = None, None
    m = re.search(r'/judgment/([a-z]+)/(\d{4})/', url)
    if m:
        code_, year = m.group(1), int(m.group(2))
    else:
        m2 = re.search(r'\[(\d{4})\]\s*(KE[A-Z]+)', title)
        if m2:
            year, code_ = int(m2.group(1)), m2.group(2).lower()
    if year is None and 'date' in row and pd.notna(row['date']):
        y = re.search(r'(19|20)\d{2}', str(row['date']))
        year = int(y.group(0)) if y else None
    d = re.search(r'@(\d{4}-\d{2}-\d{2})', url)
    if year is not None and not (1930 <= year <= 2026):
        year = None                                           # impossible years = parsing noise
    return pd.Series({'court_code': code_, 'year': year, 'decision_date': d.group(1) if d else None})

def practice_area(row):
    if row['court_code'] in COURT_AREA:
        return pd.Series({'area': COURT_AREA[row['court_code']], 'area_source': 'court'})
    t = str(row['title']).lower()
    for area, pat in TITLE_RULES:
        if re.search(pat, t):
            return pd.Series({'area': area, 'area_source': 'title'})
    return pd.Series({'area': 'Unclassified', 'area_source': 'none'})

SW = {'na','ya','wa','kwa','katika','ni','hii','kuwa','za','la','cha','hiyo','yake','lakini','mahakama','shauri','hakimu','mshtakiwa'}
EN = {'the','of','and','to','in','that','is','for','was','court','be','by'}
def language(text):
    w = re.findall(r'[a-z]+', text[:6000].lower())
    return 'Swahili' if sum(x in SW for x in w) > sum(x in EN for x in w) else 'English'

docs = pd.concat([docs, docs.apply(court_year_date, axis=1)], axis=1)
docs['court'] = docs['court_code'].map(COURTS).fillna(docs['court_code'].str.upper()).fillna('Unknown')
docs = pd.concat([docs, docs.apply(practice_area, axis=1)], axis=1)
docs.loc[docs['type'] != 'case_law', ['area', 'area_source']] = ['Legislation', 'n/a']
docs['language'] = docs['text'].apply(language)
docs[['title', 'type', 'court', 'year', 'area', 'area_source', 'language', 'n_words']].head(8)

**Advocate extractor (Base Model, stage B).** Kenyan judgments record appearances in a fairly fixed style, e.g. *"Mr. Otieno for the claimant"*, *"Ms. Wanjiku h/b for Mr. Kamau for the 1st respondent"*, *"Kamau & Co. Advocates for the plaintiff"*.
We match those patterns and keep the **party represented**, because counsel *"for the State / Republic"* are public prosecutors — a client cannot hire them, so they are removed before ranking.

In [ ]:
PARTY = (r"(?i:((?:\d+(?:st|nd|rd|th)\s+)?(?:appellants?|respondents?|applicants?|claimants?|plaintiffs?|defendants?|"
         r"petitioners?|accused(?: persons?)?|interested part(?:y|ies)|state|republic|prosecution|complainants?|"
         r"objectors?|protestors?|interveners?|decree[- ]holders?|judgment[- ]debtors?|employers?|grievants?|union)))")
STATE_ROLES = ('state', 'republic', 'prosecution')
TITLE = r"(?:Mr|Mrs|Ms|Miss|Dr|Prof|Professor)"
NAME = r"((?:[A-Z][A-Za-z'\-]+)(?:\s+(?:[A-Z][A-Za-z'\-]+|[A-Z]\.?)){0,3})"
P_COUNSEL = re.compile(TITLE + r"\s+" + NAME + r"(?:\s*,?\s*S\.?C\.?)?[^.;\n]{0,60}?\b(?i:for\s+the)\s+" + PARTY)
P_HB = re.compile(r"(?:h/b|holding brief)\s+for\s+" + TITLE + r"\s+" + NAME + r"[^.;\n]{0,40}?\b(?i:for\s+the)\s+" + PARTY)
P_FIRM = re.compile(r"(?:M/s\.?\s+)?((?:[A-Z][A-Za-z'\-]+(?:,|\s)+(?:&\s+|and\s+)?){1,5}(?:Co\.?\s*|Company\s+|Associates\s+|Partners\s+|LLP\s+)?)Advocates\b")
FIRM_STOP = {'the','learned','counsel','kenya','law','society','act','for','both','their','his','her','respective','said','senior','all','by','instructed','other','two'}
def _clean_name(n):
    n = re.sub(r"\s+", " ", n).strip(" ,.&")
    return re.sub(r"\s+S\.?C\.?$", "", n)          # 'Ojienda SC' -> 'Ojienda' (SC = Senior Counsel title)
def normalise_key(n):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z ]", " ", n.lower())).strip()
def extract_advocates(text):
    t = re.sub(r"\b(Mr|Mrs|Ms|Dr|Prof)\.", r"\1", text)   # 'Mr.' -> 'Mr' so the period isn't read as a sentence end
    found = []
    for pat, kind in ((P_COUNSEL, 'counsel'), (P_HB, 'counsel')):
        for m in pat.finditer(t):
            name = _clean_name(m.group(1)); role = re.sub(r'^\d+(st|nd|rd|th)\s+', '', m.group(2).lower()).rstrip('s')
            found.append({'name': name, 'key': normalise_key(name), 'kind': kind,
                          'role': 'state' if any(r in role for r in STATE_ROLES) else role})
    for m in P_FIRM.finditer(t):
        name = _clean_name(m.group(1))
        words = [w for w in re.findall(r"[A-Za-z']+", name)]
        while words and words[0].lower() in FIRM_STOP: words = words[1:]
        if not words or all(w.lower() in FIRM_STOP | {'co','company','associates','partners','llp','and'} for w in words): continue
        name = re.sub(r"^(?:(?:%s)\s+)+" % "|".join(FIRM_STOP), "", name, flags=re.I).strip(" ,&")
        found.append({'name': name + ' Advocates', 'key': normalise_key(name), 'kind': 'firm', 'role': 'firm'})
    seen=set(); out=[]
    for f in found:
        if f['key'] and (f['key'], f['kind']) not in seen:
            seen.add((f['key'], f['kind'])); out.append(f)
    return out


# Quick unit test on hand-written lines (covers holding brief, Senior Counsel, firms, state counsel)
demo = ("Mr. Otieno for the claimant. Ms Wanjiku h/b for Mr. Kamau Njoroge for the 1st respondent. "
        "Kamau & Co. Advocates for the Plaintiff. Ms. Jeptoo for the State. Prof. Ojienda SC for the petitioner.")
pd.DataFrame(extract_advocates(demo))

In [ ]:
cases = docs[docs['type'] == 'case_law'].copy()
cases['advocates'] = cases['text'].apply(extract_advocates)
cases['n_private_advocates'] = cases['advocates'].apply(lambda a: sum(x['role'] != 'state' for x in a))
print(f"Judgments: {len(cases):,} | with at least one private advocate found: {(cases['n_private_advocates'] > 0).mean():.1%}")

### 2.7 Exploratory Data Analysis — observations
Each code cell prints its finding with the actual numbers from the data; the note underneath explains **what we did about it**.

In [ ]:
# OBSERVATION 1 - corpus composition
comp = pd.DataFrame({'chunks': chunks['type'].value_counts(), 'documents': docs['type'].value_counts()}).fillna(0).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, col in zip(axes, comp.columns):
    ax.bar(comp.index, comp[col], color=PALETTE[0], width=0.55)
    ax.set_title(f'{col.title()} by type'); ax.set_ylabel(col)
    for i, v in enumerate(comp[col]): ax.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()
comp_ = comp.reindex(['case_law', 'legislation']).fillna(0).astype(int)
cl, lg = comp_.loc['case_law'], comp_.loc['legislation']
print(f"OBSERVATION 1: {cl['documents']:,} judgments ({cl['chunks']:,} chunks) vs {lg['documents']:,} Acts ({lg['chunks']:,} chunks). "
      f"Acts are long, so they take {lg['chunks'] / max(comp['chunks'].sum(), 1):.0%} of chunks while being "
      f"{lg['documents'] / max(comp['documents'].sum(), 1):.0%} of documents.")

*What we did:* only **case law** names advocates, so legislation is excluded from the lawyer-matching index (it can still serve a later "explain the law" feature). Mixing them would let long Acts crowd out judgments in search results.

In [ ]:
# OBSERVATION 2 - chunk length vs. what the model can actually read
MODEL_WORD_BUDGET = 190     # ~256 word-piece tokens (MiniLM's limit) ~= 190 English words; checked exactly in 3.1
wc = chunks['chunk_words']
unseen = np.clip(wc - MODEL_WORD_BUDGET, 0, None).sum() / wc.sum()
plt.figure(figsize=(8, 3.2))
plt.hist(wc, bins=60, color=PALETTE[0])
plt.axvline(MODEL_WORD_BUDGET, color='#333', ls='--', lw=1.5); plt.text(MODEL_WORD_BUDGET, plt.ylim()[1]*0.9, '  MiniLM limit (~190 words)', fontsize=9)
plt.title('Words per original chunk'); plt.xlabel('words'); plt.ylabel('chunks'); plt.show()
print(f"OBSERVATION 2: median chunk = {wc.median():.0f} words; {(wc > MODEL_WORD_BUDGET).mean():.0%} of chunks are longer than the model's input limit. "
      f"Embedding the chunks as provided would silently ignore {unseen:.0%} of all text.")

*What we did:* this is the most important finding for modelling. A transformer **truncates** anything past its limit, so the facts in the middle and end of an 800-word chunk would never influence search. We rebuild each judgment (2.5) and re-split it into **160-word passages with a 30-word overlap** (2.8), then confirm with the model's own tokenizer in 3.1.

In [ ]:
# OBSERVATION 3 - is the advertised 100-word overlap real?
if len(overlaps):
    has = overlaps > 0
    print(f"OBSERVATION 3: {has.mean():.0%} of neighbouring chunk pairs share an exact overlap; "
          f"median overlap = {np.median(overlaps[has]) if has.any() else 0:.0f} words. "
          f"Removing it cut {chunks['chunk_words'].sum() - docs['n_words'].sum():,} duplicated words "
          f"({1 - docs['n_words'].sum() / chunks['chunk_words'].sum():.1%} of the corpus).")
else:
    print("OBSERVATION 3: every document is a single chunk, so there is no overlap to remove.")

*What we did:* the overlap is removed before re-splitting, so no sentence (and no advocate name) is counted twice.

In [ ]:
# OBSERVATION 4 - practice areas and courts (class balance + how many labels we could recover)
area_counts = cases['area'].value_counts()
court_counts = cases['court'].value_counts().head(10)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].barh(area_counts.index[::-1], area_counts.values[::-1], color=PALETTE[0]); axes[0].set_title('Judgments per practice area')
axes[1].barh(court_counts.index[::-1], court_counts.values[::-1], color=PALETTE[0]); axes[1].set_title('Judgments per court (top 10)')
plt.tight_layout(); plt.show()
labelled = area_counts.drop(['General Civil', 'Unclassified'], errors='ignore')
print(f"OBSERVATION 4: a reliable practice area was recovered for {labelled.sum() / len(cases):.0%} of judgments "
      f"({(cases['area_source'] == 'court').mean():.0%} from a specialised court, {(cases['area_source'] == 'title').mean():.0%} from the title). "
      f"Largest area = {labelled.idxmax() if len(labelled) else '-'} ({labelled.max() if len(labelled) else 0:,}), "
      f"smallest = {labelled.idxmin() if len(labelled) else '-'} ({labelled.min() if len(labelled) else 0:,}) -> "
      f"imbalance ratio {labelled.max() / max(labelled.min(), 1):.1f}:1. Court missing for {(cases['court'] == 'Unknown').mean():.0%}.")

*What we did:* the areas are **imbalanced**, so (a) the modelling sample is *stratified* — capped at the same number of judgments per area — (b) cross-validation uses *stratified* folds, and (c) we report **macro-F1** (every area counts equally) instead of plain accuracy, which a model could inflate by favouring the biggest area.

In [ ]:
# OBSERVATION 5 - how recent are the judgments?
yrs = cases['year'].dropna().astype(int)
plt.figure(figsize=(9, 3.2))
vc = yrs.value_counts().sort_index()
plt.bar(vc.index, vc.values, color=PALETTE[0], width=0.8)
plt.title('Judgments per year of decision'); plt.xlabel('year'); plt.ylabel('judgments'); plt.show()
print(f"OBSERVATION 5: year recovered for {cases['year'].notna().mean():.0%} of judgments; range {yrs.min() if len(yrs) else '-'}-{yrs.max() if len(yrs) else '-'}; "
      f"{(yrs >= 2015).mean():.0%} were decided in 2015 or later; median year {yrs.median() if len(yrs) else float('nan'):.0f}.")

*What we did:* recency matters for matching — an advocate who appeared in 2023 is far more likely to still be practising than one from 1998. The year is kept as a feature so the app can prefer recent cases, and so we can later check names against the current LSK register (Q4).

In [ ]:
# OBSERVATION 6 - language
lang = cases['language'].value_counts()
print(f"OBSERVATION 6: {lang.get('Swahili', 0):,} judgments ({lang.get('Swahili', 0) / len(cases):.1%}) are mainly in Swahili; the rest are English.")

*What we did:* our base embedder was trained on English, so Swahili judgments are **excluded from this base model** rather than being embedded badly. They are kept on disk; a multilingual model (e.g. `paraphrase-multilingual-MiniLM-L12-v2`) is a planned extension, since many clients may describe their issue in Swahili or Sheng.

In [ ]:
# OBSERVATION 7 - can we actually find lawyers in the text?
adv_flat = [a for lst in cases['advocates'] for a in lst]
roles = Counter(a['role'] for a in adv_flat)
top_names = Counter(a['name'] for a in adv_flat if a['kind'] == 'counsel' and a['role'] != 'state').most_common(10)
single_word = np.mean([len(a['key'].split()) == 1 for a in adv_flat if a['kind'] == 'counsel']) if adv_flat else 0
print(f"OBSERVATION 7: at least one private advocate found in {(cases['n_private_advocates'] > 0).mean():.0%} of judgments "
      f"(median {cases.loc[cases['n_private_advocates'] > 0, 'n_private_advocates'].median() if (cases['n_private_advocates'] > 0).any() else 0:.0f} per judgment). "
      f"{roles.get('state', 0):,} mentions are state counsel (removed). {single_word:.0%} of counsel are recorded by surname only.")
print("Most frequent advocate names:", top_names)

*What we did:* coverage is good enough to build matching on, but two risks are visible: judgments with **no** advocate found cannot contribute a lawyer, and **surname-only** names (*"Mr. Kamau"*) are ambiguous — there are many advocates called Kamau. This is why Q4 includes checking extracted names against the Law Society of Kenya register before showing them to a client.

### 2.8 Final modelling dataset
Filters: case law only · English only · at least 50 words. Each judgment is then re-split into model-sized passages (passages keep their `doc_id`, so search results can be grouped back into cases).

In [ ]:
before = len(docs)
final_cases = cases[(cases['language'] == 'English')].copy()
log_step('Keep English case law only', before, len(final_cases), 'Lawyers appear only in judgments; base embedder is English-only')

def make_passages(words, size, overlap):
    step, out = size - overlap, []
    for start in range(0, max(len(words) - overlap, 1), step):
        piece = words[start:start + size]
        if len(piece) < 25 and out:      # a tiny tail is already covered by the previous passage's overlap
            break
        out.append(' '.join(piece))
    return out

rows = []
for r in final_cases.itertuples():
    for i, p in enumerate(make_passages(r.text.split(), CONFIG['PASSAGE_WORDS'], CONFIG['PASSAGE_OVERLAP'])):
        rows.append({'doc_id': r.doc_id, 'passage_no': i, 'text': p, 'area': r.area, 'court': r.court, 'year': r.year})
passages_all = pd.DataFrame(rows)
log_step('Re-split judgments into passages', len(final_cases), len(passages_all), '(rows grow: one judgment -> many passages)')

print("\nFinal judgments:", final_cases.shape, "| Final passages:", passages_all.shape)
passages_all.head(3)

In [ ]:
# Cleaning summary: the full path from raw rows to modelling rows
print("Missing/empty % before vs after cleaning (judgment-level fields):")
after = pd.Series({'title': is_blank(final_cases['title']).mean() * 100,
                   'court': (final_cases['court'] == 'Unknown').mean() * 100,
                   'year': final_cases['year'].isna().mean() * 100}).round(2)
print(after.to_string())
pd.DataFrame(cleaning_log)

## 3. Modelling Approach — Base Model (Q3)

**Base model:** `all-MiniLM-L6-v2` sentence-transformer. It is *pre-trained*, so "training" here means **building the index** (turning every passage into a 384-number vector). The client's text is turned into a vector the same way, and we rank passages by **cosine similarity** (how closely the two vectors point in the same direction).

**How we check that the results are valid** (the retrieval equivalent of the elbow method / cross-validation / regularisation):

| Check | Question it answers | Section |
|---|---|---|
| Tokenizer check | Does the model actually read our whole passage? | 3.1 |
| Labelled client-query test set → Precision@K, MRR, nDCG vs a random baseline | When a client describes an employment problem, are the top cases employment cases? | 3.4 |
| Stratified **group** 5-fold **cross-validation** of a **regularised** logistic regression on the embeddings, with a validation curve over C | Do the embeddings separate legal areas on judgments the model has never seen — and is that stable across folds? | 3.5 |
| Similarity-threshold analysis (ROC) | When should the app say "no good match" instead of suggesting a random lawyer? | 3.6 |
| Extractor coverage + manual labelling template | Are the extracted lawyer names right? | 3.7 |

### 3.1 Load the model and confirm passage size

In [ ]:
embedder = SentenceTransformer(CONFIG['BASE_MODEL'])
max_len = embedder.max_seq_length
print("Model max input length (word-piece tokens):", max_len)

def token_lengths(texts):
    return np.array([len(ids) for ids in embedder.tokenizer(list(texts), add_special_tokens=True)['input_ids']])

s_chunks = chunks.loc[chunks['type'] == 'case_law', 'text'].sample(min(300, (chunks['type'] == 'case_law').sum()), random_state=SEED)
s_pass = passages_all['text'].sample(min(300, len(passages_all)), random_state=SEED)
tl_c, tl_p = token_lengths(s_chunks), token_lengths(s_pass)
print(f"Original chunks : median {np.median(tl_c):.0f} tokens, {(tl_c > max_len).mean():.0%} truncated")
print(f"Our passages    : median {np.median(tl_p):.0f} tokens, {(tl_p > max_len).mean():.0%} truncated")

If more than ~10% of our passages were still truncated we would lower `PASSAGE_WORDS`; the printout confirms the re-splitting from 2.8 works.

### 3.2 Build a stratified modelling sample
Encoding every passage of every judgment is an offline job for the production app. To keep this notebook runnable in class we take an **equal number of judgments per practice area** (`MAX_DOCS_PER_AREA`), which also prevents the largest area from dominating the evaluation.

In [ ]:
sample_docs = pd.concat([g.sample(min(len(g), CONFIG['MAX_DOCS_PER_AREA']), random_state=SEED)
                         for _, g in final_cases.groupby('area')])
passages = (passages_all[passages_all['doc_id'].isin(sample_docs['doc_id'])]
            .groupby('doc_id', group_keys=False).head(CONFIG['MAX_PASSAGES_PER_DOC'])
            .reset_index(drop=True))
doc_area = sample_docs.set_index('doc_id')['area']
doc_title = sample_docs.set_index('doc_id')['title']
advocates_by_doc = sample_docs.set_index('doc_id')['advocates'].to_dict()
print(f"Sample: {len(sample_docs):,} judgments -> {len(passages):,} passages")
print(sample_docs['area'].value_counts().to_string())

### 3.3 Build the index and the matching pipeline

In [ ]:
t0 = time.time()
P_emb = embedder.encode(passages['text'].tolist(), batch_size=64, show_progress_bar=True,
                        normalize_embeddings=True, convert_to_numpy=True)   # unit-length vectors -> dot product = cosine
base_encode_time = time.time() - t0
print(f"Encoded {P_emb.shape[0]:,} passages into {P_emb.shape[1]}-d vectors in {base_encode_time:.1f}s")

pass_doc = passages['doc_id'].to_numpy(dtype=str)

def rank_cases(passage_scores, top_n=20):
    # a judgment's score = its best-matching passage (one strong paragraph about the same issue is what matters)
    s = pd.Series(passage_scores).groupby(pass_doc).max()
    return s.sort_values(ascending=False).head(top_n)

def dense_scores(query, model=embedder, emb=None):
    emb = P_emb if emb is None else emb
    q = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]
    return emb @ q

def match_lawyers(query, top_cases=20, top_lawyers=5):
    best = rank_cases(dense_scores(query), top_cases)
    rows = []
    for doc_id, score in best.items():
        for a in advocates_by_doc.get(doc_id, []):
            if a['role'] == 'state':
                continue              # public prosecutors cannot be hired
            rows.append({'lawyer': a['name'], 'key': a['key'], 'score': score,
                         'case': doc_title.get(doc_id, doc_id), 'area': doc_area.get(doc_id)})
    if not rows:
        return best, pd.DataFrame()
    r = pd.DataFrame(rows)
    ranked = (r.groupby('key').agg(lawyer=('lawyer', 'first'), match_score=('score', 'sum'),
                                  relevant_cases=('case', 'nunique'), example_case=('case', 'first'),
                                  areas=('area', lambda x: ', '.join(sorted(set(x)))))
               .sort_values('match_score', ascending=False).head(top_lawyers).reset_index(drop=True))
    return best, ranked

q = "My employer fired me without any notice or a hearing after I reported an injury at work"
best, lawyers = match_lawyers(q)
print("CLIENT:", q, "\n\nTop matching judgments:")
print(pd.DataFrame({'similarity': best.head(5).round(3).values, 'area': doc_area.reindex(best.head(5).index).values,
                    'title': doc_title.reindex(best.head(5).index).str[:90].values}).to_string(index=False))
print("\nSuggested lawyers:")
lawyers

### 3.4 Validation A — labelled client-query test set
There are no ready-made relevance labels for Kenyan legal search, so we wrote **35 realistic client queries in plain language** (5 per practice area). A retrieved judgment counts as *relevant* when its practice area matches the query's area.

* **Precision@5 / @10** – share of the top 5 / 10 judgments that are relevant.
* **MRR** (mean reciprocal rank) – 1 if the first result is relevant, ½ if the first relevant one is 2nd, …
* **nDCG@10** – like precision, but rewards relevant results that appear higher.
* **Random baseline** – the precision you'd get by picking judgments at random (= the area's share of the sample). A useful model must clearly beat it.

*Limitation (stated honestly):* area-level relevance is coarse — a land judgment about boundaries is "relevant" to a query about eviction. So these numbers are an **upper-level sanity check**; a lawyer reviewing the top results (Q4) gives finer relevance.

In [ ]:
EVAL_QUERIES = [
 ("I was dismissed from my job without notice and they refused to pay my terminal dues", "Employment & Labour"),
 ("My boss has not paid my salary for three months and now says I resigned", "Employment & Labour"),
 ("I was declared redundant but other people were kept on and I got no package", "Employment & Labour"),
 ("I was injured at the factory and the company says it is not responsible", "Employment & Labour"),
 ("They demoted me and cut my pay after I joined the union", "Employment & Labour"),
 ("Someone has fenced off part of my plot and claims the land is theirs", "Land & Property"),
 ("The seller took my money for a plot but refuses to transfer the title deed", "Land & Property"),
 ("My landlord locked my shop because I was late with rent", "Land & Property"),
 ("The county wants to evict us from land our family has lived on for forty years", "Land & Property"),
 ("There are two title deeds for the same piece of land and the other owner is building", "Land & Property"),
 ("My son was arrested and charged with robbery but he was at home that night", "Criminal"),
 ("The police are holding my brother and refusing to release him on bail", "Criminal"),
 ("I was convicted of stealing and want to appeal the sentence", "Criminal"),
 ("I am accused of defilement and I need to defend myself in court", "Criminal"),
 ("A relative was charged with murder after a fight at a bar", "Criminal"),
 ("My father died without a will and my uncles are taking his property", "Family & Succession"),
 ("I want a divorce and custody of our two children", "Family & Succession"),
 ("My husband sold our matrimonial home without my consent", "Family & Succession"),
 ("My stepmother is the administrator of the estate and is excluding us", "Family & Succession"),
 ("The father of my child refuses to pay child maintenance", "Family & Succession"),
 ("The government closed our business without giving us a chance to be heard", "Constitutional & Administrative"),
 ("A public body made a decision affecting me without following the law and I want it reviewed", "Constitutional & Administrative"),
 ("My constitutional rights were violated when officers raided my home", "Constitutional & Administrative"),
 ("The county revoked my licence unfairly and without reasons", "Constitutional & Administrative"),
 ("I want to challenge a new law that discriminates against people like me", "Constitutional & Administrative"),
 ("A supplier took a deposit and never delivered the goods in our contract", "Commercial & Contract"),
 ("The bank is auctioning my property over a loan I have been repaying", "Commercial & Contract"),
 ("My business partner breached our agreement and took the clients", "Commercial & Contract"),
 ("A company owes us money for work done and is now being wound up", "Commercial & Contract"),
 ("We signed a contract with an arbitration clause and the other side refuses to pay", "Commercial & Contract"),
 ("KRA has sent me a huge tax assessment that I think is wrong", "Tax"),
 ("We were charged VAT on goods that should be exempt", "Tax"),
 ("The tax authority froze our bank accounts over unpaid income tax", "Tax"),
 ("I want to object to the penalties and interest added to my tax bill", "Tax"),
 ("Customs valued our imported goods too high and charged extra duty", "Tax"),
]
eval_df = pd.DataFrame(EVAL_QUERIES, columns=['query', 'area'])
present = set(doc_area.unique())
dropped = sorted(set(eval_df['area']) - present)
eval_df = eval_df[eval_df['area'].isin(present)].reset_index(drop=True)
if dropped:
    print("Areas with no judgments in the sample (their queries are skipped):", dropped)
print(f"{len(eval_df)} evaluation queries across {eval_df['area'].nunique()} areas")

def metrics_for(ranked_doc_ids, true_area, k_list=(5, 10)):
    rel = np.array([doc_area.get(d) == true_area for d in ranked_doc_ids], dtype=float)
    out = {f'P@{k}': rel[:k].mean() if len(rel) else 0 for k in k_list}
    hits = np.flatnonzero(rel)
    out['MRR'] = 1 / (hits[0] + 1) if len(hits) else 0.0
    gains = rel[:10] / np.log2(np.arange(2, min(10, len(rel)) + 2))
    n_rel = (doc_area == true_area).sum()            # relevant judgments that exist in the index
    ideal = np.ones(min(10, n_rel)) / np.log2(np.arange(2, min(10, n_rel) + 2))
    out['nDCG@10'] = gains.sum() / ideal.sum() if ideal.sum() else 0
    return out

def evaluate(score_fn, name):
    rows = []
    for r in eval_df.itertuples():
        ranked = rank_cases(score_fn(r.query), top_n=20).index
        rows.append({'model': name, 'query': r.query, 'area': r.area, **metrics_for(ranked, r.area)})
    return pd.DataFrame(rows)

res_base = evaluate(dense_scores, 'MiniLM-L6 (base)')
prevalence = doc_area.value_counts(normalize=True)
random_p = eval_df['area'].map(prevalence).mean()

summary = res_base[['P@5', 'P@10', 'MRR', 'nDCG@10']].mean().round(3)
print("Base model (mean over queries):"); print(summary.to_string())
print(f"Random baseline P@K = {random_p:.3f}  ->  lift at P@5 = {summary['P@5'] / random_p:.1f}x")
res_base.groupby('area')[['P@5', 'MRR']].mean().round(2).sort_values('P@5')

*Reading it:* the per-area table shows **where** the model struggles — typically areas whose language overlaps (e.g. Constitutional vs Land evictions, Commercial vs General Civil). Those areas get priority for the improvements in Q4.

### 3.5 Validation B — cross-validation with regularisation
**Idea:** if the embeddings capture legal meaning, a simple linear model should be able to tell the practice area of a passage **it has never seen**.

Two details make this check honest:
1. **Grouped folds (`StratifiedGroupKFold`).** Passages from the *same judgment* are very similar. If some went into training and others into testing, the model would "recognise" the case instead of the area — that is *data leakage*. Grouping by `doc_id` keeps each judgment entirely in one fold. We also run the naive (ungrouped) version to show how much leakage would have inflated the score.
2. **Regularisation (L2, parameter C).** Small C = strong penalty on large weights = simpler model. We try several values and plot a **validation curve**: training score vs cross-validated score. A large gap means over-fitting; we pick the C with the best cross-validated macro-F1.

In [ ]:
lab_mask = ~passages['area'].isin(['General Civil', 'Unclassified']).to_numpy()
area_docs = passages.loc[lab_mask].groupby('area')['doc_id'].nunique()
keep_areas = area_docs[area_docs >= 10].index          # each fold needs a few judgments from every area
mask = lab_mask & passages['area'].isin(keep_areas).to_numpy()
X = P_emb[mask]
y = passages.loc[mask, 'area'].to_numpy(dtype=str)          # plain numpy arrays for scikit-learn
groups = passages.loc[mask, 'doc_id'].to_numpy(dtype=str)
print(f"CV data: {X.shape[0]:,} passages from {len(set(groups)):,} judgments, {len(keep_areas)} areas")

cv_group = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
C_values = [0.01, 0.1, 1, 10, 100]
curve = []
for C in C_values:
    clf = LogisticRegression(C=C, max_iter=3000, class_weight='balanced')
    tr, te = [], []
    for tr_idx, te_idx in cv_group.split(X, y, groups):
        clf.fit(X[tr_idx], y[tr_idx])
        tr.append(f1_score(y[tr_idx], clf.predict(X[tr_idx]), average='macro'))
        te.append(f1_score(y[te_idx], clf.predict(X[te_idx]), average='macro'))
    curve.append({'C': C, 'train_F1': np.mean(tr), 'cv_F1': np.mean(te), 'cv_F1_std': np.std(te)})
curve = pd.DataFrame(curve)
best_C = curve.loc[curve['cv_F1'].idxmax(), 'C']

plt.figure(figsize=(6.5, 3.6))
plt.plot(curve['C'], curve['train_F1'], marker='o', lw=2, color=PALETTE[1], label='training')
plt.errorbar(curve['C'], curve['cv_F1'], yerr=curve['cv_F1_std'], marker='o', lw=2, capsize=4, color=PALETTE[0], label='5-fold CV (unseen judgments)')
plt.xscale('log'); plt.xlabel('C  (smaller = stronger regularisation)'); plt.ylabel('macro-F1')
plt.title('Validation curve: logistic regression on MiniLM embeddings'); plt.legend(); plt.show()
print(curve.round(3).to_string(index=False)); print("Best C:", best_C)

In [ ]:
best_clf = LogisticRegression(C=best_C, max_iter=3000, class_weight='balanced')
grouped = cross_val_score(best_clf, X, y, groups=groups, cv=cv_group, scoring='f1_macro')
naive = cross_val_score(best_clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), scoring='f1_macro')
print(f"Grouped CV macro-F1 : {grouped.mean():.3f} +/- {grouped.std():.3f}   (per fold: {np.round(grouped, 3)})")
print(f"Naive CV macro-F1   : {naive.mean():.3f}   -> leakage would have inflated the score by {naive.mean() - grouped.mean():+.3f}")
print(f"Chance level (macro-F1 of always guessing evenly) ~ {1 / len(keep_areas):.3f}")

pred = cross_val_predict(best_clf, X, y, groups=groups, cv=cv_group)
labels = sorted(keep_areas)
fig, ax = plt.subplots(figsize=(7.5, 6))
ConfusionMatrixDisplay(confusion_matrix(y, pred, labels=labels, normalize='true'), display_labels=labels).plot(
    ax=ax, cmap='Blues', values_format='.2f', colorbar=False)
ax.grid(False); plt.xticks(rotation=45, ha='right'); plt.title('Cross-validated confusion matrix (row-normalised)'); plt.tight_layout(); plt.show()
print(classification_report(y, pred, digits=3))

*Reading it:* a small fold-to-fold standard deviation means the result is **stable**, not luck of one split. The confusion matrix shows which areas the embeddings mix up — usually the same pairs that scored low in 3.4, which is a good sign the two checks agree.

### 3.6 Validation C — when should the app say "no good match"?
Cosine similarity always returns *something*, even for "how do I cook pilau". We compare the best similarity score for genuine legal queries against off-topic ones and choose the cut-off that best separates them (Youden's J on the ROC curve).

In [ ]:
OFF_TOPIC = ["How do I cook pilau for ten people", "Which phone has the best camera under 30,000 shillings",
             "What time does the Arsenal match start on Saturday", "How do I fix a slow laptop",
             "Recommend a good novel to read on holiday", "How do I grow sukuma wiki in a small garden",
             "What is the weather in Mombasa next week", "How do I make my matatu playlist louder"]
top1_legal = np.array([dense_scores(q).max() for q in eval_df['query']])
top1_off = np.array([dense_scores(q).max() for q in OFF_TOPIC])
y_true = np.r_[np.ones(len(top1_legal)), np.zeros(len(top1_off))]
scores = np.r_[top1_legal, top1_off]
fpr, tpr, thr = roc_curve(y_true, scores)
THRESHOLD = float(thr[np.argmax(tpr - fpr)])
plt.figure(figsize=(7, 3.2))
plt.hist(top1_legal, bins=15, alpha=0.8, color=PALETTE[0], label='legal queries')
plt.hist(top1_off, bins=15, alpha=0.8, color=PALETTE[1], label='off-topic queries')
plt.axvline(THRESHOLD, color='#333', ls='--'); plt.xlabel('best cosine similarity'); plt.legend(); plt.title('Top-1 similarity'); plt.show()
print(f"AUC = {roc_auc_score(y_true, scores):.3f}; chosen threshold = {THRESHOLD:.3f} -> below this the app asks the client to rephrase.")

### 3.7 Validation D — the lawyer extractor
Retrieval can be perfect and still show the wrong lawyer if the names are extracted badly, so the extractor is checked separately:
* **Coverage** (automatic): share of matched judgments that yield at least one hireable advocate.
* **Precision / recall / F1** (needs ground truth): we export 50 random judgments; team members open each one on Kenya Law and type the true advocate names into `true_advocates`. Running the next cell again then scores the extractor. *(Instructions say a description is enough at this stage; the template makes the final-stage evaluation one step.)*

In [ ]:
cov = np.mean([any(a['role'] != 'state' for a in advocates_by_doc.get(d, [])) for d in rank_cases(dense_scores(q), 20).index])
print(f"Coverage: {cov:.0%} of the top-20 judgments for the demo query name a hireable advocate")

LABEL_FILE = 'artifacts/ner_labelling_sample.csv'
if not os.path.exists(LABEL_FILE):
    lab = sample_docs.sample(min(50, len(sample_docs)), random_state=SEED)
    pd.DataFrame({'doc_id': lab['doc_id'], 'title': lab['title'], 'url': lab['url'],
                  'predicted_advocates': lab['advocates'].apply(lambda a: '; '.join(x['name'] for x in a if x['role'] != 'state')),
                  'true_advocates': ''}).to_csv(LABEL_FILE, index=False)
    print(f"Labelling template written to {LABEL_FILE} -> fill 'true_advocates' (semicolon-separated) and re-run.")

def score_extractor(path=LABEL_FILE):
    lab = pd.read_csv(path).fillna('')
    lab = lab[lab['true_advocates'].str.strip() != '']
    if lab.empty:
        return None
    tp = fp = fn = 0
    for r in lab.itertuples():
        p = {normalise_key(x) for x in r.predicted_advocates.split(';') if x.strip()}
        t = {normalise_key(x) for x in r.true_advocates.split(';') if x.strip()}
        tp += len(p & t); fp += len(p - t); fn += len(t - p)
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {'judgments': len(lab), 'precision': prec, 'recall': rec, 'F1': 2 * prec * rec / max(prec + rec, 1e-9)}

print("Extractor scores:", score_extractor() or "waiting for manual labels")

In [ ]:
# Save what the application needs: passages, their vectors, advocates per judgment, and the threshold
passages.to_parquet('artifacts/passages.parquet', index=False)
np.save('artifacts/passage_embeddings.npy', P_emb.astype(np.float32))
json.dump({'model': CONFIG['BASE_MODEL'], 'threshold': THRESHOLD, 'config': CONFIG,
           'advocates_by_doc': advocates_by_doc}, open('artifacts/engine_meta.json', 'w'), default=str)
print("Saved:", os.listdir('artifacts'))

### 3.8 Base model — summary of the approach
1. **Data:** English case law, rebuilt from overlapping chunks and re-split into 160-word passages that fit the model.
2. **Model:** `all-MiniLM-L6-v2` turns passages and client queries into vectors; cosine similarity ranks passages; each judgment takes its best passage's score.
3. **Lawyers:** advocates are extracted from the top 20 judgments, state counsel removed, and ranked by the summed similarity of the cases they argued.
4. **Validity:** retrieval metrics against a random baseline (3.4), grouped & stratified 5-fold cross-validation with an L2 regularisation curve (3.5), a no-match threshold (3.6), and extractor coverage plus a labelling protocol (3.7).

## 4. Further Validation — comparing models through the same pipeline (Q4)

Following the recommended approach, we run **several alternative retrieval models through exactly the same process** (same passages, same 35 queries, same case-level ranking, same metrics) and compare them side by side. If our base model is genuinely good, it should beat the keyword baselines; if a bigger model is much better, that tells us what to deploy.

| Model | Type | Why include it |
|---|---|---|
| TF-IDF + cosine | keyword (sparse) | classic baseline: are embeddings worth the cost? |
| BM25 | keyword (probabilistic) | the standard search-engine baseline; strong on exact legal terms |
| **MiniLM-L6 (base)** | dense | our base model |
| multi-qa-MiniLM-L6 | dense | same size, but trained specifically on *question → answer passage* pairs, which is our situation |
| all-mpnet-base-v2 | dense | ~5× bigger; is the accuracy gain worth the speed cost? |
| Hybrid BM25 + MiniLM (RRF) | fusion | combines exact-term matching with meaning; often best in legal search |

**Same metrics for all:** P@5, MRR, nDCG@10 (3.4), cross-validated macro-F1 where the model produces vectors (3.5), and encoding time.
**Is a difference real?** We use a **paired bootstrap** over the 35 queries: resample the queries 2,000 times and see whether the MRR difference against the base model stays above zero in 95% of resamples.

In [ ]:
results, timing, cv_scores = [res_base], {'MiniLM-L6 (base)': base_encode_time}, {}
texts = passages['text'].tolist()

def grouped_cv_f1(Xm):
    clf = LogisticRegression(C=best_C, max_iter=3000, class_weight='balanced')
    return cross_val_score(clf, Xm[mask], y, groups=groups, cv=cv_group, scoring='f1_macro').mean()

cv_scores['MiniLM-L6 (base)'] = grouped.mean()

# --- TF-IDF ---
t0 = time.time()
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=60000, sublinear_tf=True, stop_words='english')
T = tfidf.fit_transform(texts)
timing['TF-IDF'] = time.time() - t0
results.append(evaluate(lambda q: (T @ tfidf.transform([q]).T).toarray().ravel(), 'TF-IDF'))
cv_scores['TF-IDF'] = grouped_cv_f1(T.tocsr())

# --- BM25 ---
tok = lambda s: re.findall(r'[a-z0-9]+', s.lower())
t0 = time.time(); bm25 = BM25Okapi([tok(t) for t in texts]); timing['BM25'] = time.time() - t0
bm25_scores = lambda q: bm25.get_scores(tok(q))
results.append(evaluate(bm25_scores, 'BM25'))

# --- Hybrid: reciprocal rank fusion of BM25 and MiniLM passage rankings ---
def rrf(q, k=60):
    out = np.zeros(len(texts))
    for s in (bm25_scores(q), dense_scores(q)):
        ranks = np.empty(len(s)); ranks[np.argsort(-s)] = np.arange(1, len(s) + 1)
        out += 1 / (k + ranks)
    return out
results.append(evaluate(rrf, 'Hybrid BM25+MiniLM'))

# --- Other dense models ---
other = {'multi-qa-MiniLM-L6': 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1'}
if CONFIG['RUN_FULL_COMPARISON']:
    other['mpnet-base'] = 'sentence-transformers/all-mpnet-base-v2'
for name, path in other.items():
    m = SentenceTransformer(path)
    t0 = time.time()
    E = m.encode(texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    timing[name] = time.time() - t0
    results.append(evaluate(lambda q, m=m, E=E: dense_scores(q, m, E), name))
    cv_scores[name] = grouped_cv_f1(E)

all_res = pd.concat(results, ignore_index=True)
comparison = all_res.groupby('model')[['P@5', 'MRR', 'nDCG@10']].mean()
comparison['CV macro-F1'] = pd.Series(cv_scores)
comparison.loc['Random baseline', ['P@5']] = random_p
comparison = comparison.sort_values('MRR', ascending=False)
comparison.round(3)

In [ ]:
# Figure: same pipeline, different models, metrics side by side
plot_df = comparison.drop('Random baseline', errors='ignore')[['P@5', 'MRR', 'nDCG@10', 'CV macro-F1']]
fig, ax = plt.subplots(figsize=(11, 4.8))
n_m, n_k = plot_df.shape
w = 0.8 / n_k
for j, metric in enumerate(plot_df.columns):
    ax.bar(np.arange(n_m) + j * w - 0.4 + w / 2, plot_df[metric].fillna(0), width=w * 0.92, color=PALETTE[j], label=metric)
ax.axhline(random_p, color='#555', ls=':', lw=1.5); ax.text(n_m - 0.5, random_p, ' random P@5', va='bottom', fontsize=9)
ax.set_xticks(np.arange(n_m)); ax.set_xticklabels(plot_df.index, rotation=15)
ax.set_ylim(0, 1); ax.set_ylabel('score (higher is better)')
ax.set_title('Model comparison on the same passages, queries and metrics  (CV macro-F1 not defined for BM25 / hybrid)')
ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.2), frameon=False); plt.tight_layout(); plt.show()

In [ ]:
# Paired bootstrap: is each model's MRR really different from the base model's?
base = all_res[all_res['model'] == 'MiniLM-L6 (base)'].set_index('query')['MRR']
rows = []
for name in all_res['model'].unique():
    if name == 'MiniLM-L6 (base)':
        continue
    diff = (all_res[all_res['model'] == name].set_index('query')['MRR'] - base).loc[base.index].values
    boots = [rng.choice(diff, len(diff), replace=True).mean() for _ in range(2000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    rows.append({'model': name, 'MRR diff vs base': diff.mean(), '95% CI low': lo, '95% CI high': hi,
                 'verdict': 'better' if lo > 0 else ('worse' if hi < 0 else 'no clear difference')})
pd.DataFrame(rows).round(3)

In [ ]:
print("Encoding / indexing time for the sample (seconds):", {k: round(v, 1) for k, v in timing.items()})

### 4.1 Interpreting the comparison *(update after the final run)*
* If **MiniLM beats TF-IDF/BM25**, the meaning-based approach is justified for lay-language queries.
* If **multi-qa-MiniLM** wins, it is a drop-in replacement of the same size and speed — the cheapest improvement.
* If **mpnet** wins only by a "no clear difference" margin, the 5× slower encoding is not worth it for a mobile-backed app.
* If the **hybrid** wins, keep both indexes: BM25 catches exact statute names ("Section 45 of the Employment Act"), embeddings catch paraphrases.

### 4.2 Other validation planned for the final notebook
1. **Expert relevance judgments.** A practising advocate rates the top-5 judgments for 20 queries as relevant / partly / not relevant. This replaces our coarse area-level labels with real relevance and lets us compute graded nDCG.
2. **Extractor comparison.** Score the rule-based extractor against spaCy `en_core_web_sm` PERSON entities on the same 50 hand-labelled judgments (3.7). spaCy finds *people*, but cannot tell an advocate from a judge or a party, so we expect lower precision — the comparison quantifies it.
3. **Cross-referencing with the LSK register.** Match extracted names against the Law Society of Kenya directory to measure the *verified-match rate* (share of suggested lawyers who are real, currently practising advocates). Surname-only names (Observation 7) are matched using the firm name and court station to reduce ambiguity.
4. **Fine-tuning (if time allows).** Fine-tune MiniLM on (case summary → judgment passage) pairs with a contrastive loss, using a judgment-level train/validation/test split so no case appears in two sets, and early stopping on validation MRR.
5. **Bias check.** Compare how often lawyers from Nairobi vs other court stations are suggested, so the ranking does not just favour the busiest courts.